# 📚 RAG Pipeline — Membre 1
### Gestion des documents PDF/DOCX : chargement, découpage, vectorisation et recherche

---
| Fonction | Rôle |
|---|---|
| `charger_document()` | Lit un PDF ou DOCX et retourne le texte brut |
| `decouper_chunks()` | Découpe le texte en petits morceaux |
| `creer_base_vectorielle()` | Transforme les morceaux en vecteurs (FAISS) |
| `rechercher()` | Trouve les passages les plus pertinents |

## 📦 Cellule 1 — Installation des dépendances

!pip install pymupdf python-docx langchain-text-splitters faiss-cpu sentence-transformers -q

## 📥 Cellule 2 — Imports

In [1]:
import os
import re
from typing import List, Tuple

import fitz                          # PyMuPDF → PDF
from docx import Document            # python-docx → DOCX
from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print("✅ Tous les imports sont OK !")

✅ Tous les imports sont OK !


## 🧠 Cellule 3 — Chargement du modèle d'embedding (singleton)

In [2]:
# Modèle multilingue — fonctionne très bien en français
_EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
_embed_model = None

def _get_embed_model() -> SentenceTransformer:
    """Retourne le modèle d'embedding (chargé une seule fois)."""
    global _embed_model
    if _embed_model is None:
        print(f"⏳ Chargement du modèle : {_EMBED_MODEL_NAME}…")
        _embed_model = SentenceTransformer(_EMBED_MODEL_NAME)
        print("✅ Modèle chargé !")
    return _embed_model

# Pré-chargement
_get_embed_model()

⏳ Chargement du modèle : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2…


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé !


SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

---
## 1️⃣ Cellule 4 — `charger_document(fichier)`
Lit un PDF ou DOCX et retourne le texte brut nettoyé.

In [3]:
def _lire_pdf(chemin: str) -> str:
    """Extrait le texte de toutes les pages d'un PDF avec PyMuPDF."""
    pages = []
    with fitz.open(chemin) as doc:
        for page in doc:
            pages.append(page.get_text("text"))
    return "\n".join(pages)


def _lire_docx(chemin: str) -> str:
    """Extrait le texte de tous les paragraphes d'un DOCX."""
    doc = Document(chemin)
    return "\n".join(para.text for para in doc.paragraphs)


def _nettoyer_texte(texte: str) -> str:
    """Supprime les espaces et sauts de ligne superflus."""
    texte = re.sub(r"[ \t]+", " ", texte)      # espaces multiples → 1
    texte = re.sub(r"\n{3,}", "\n\n", texte)    # > 2 sauts → 2
    return texte.strip()


def charger_document(fichier: str) -> str:
    """
    Lit un fichier PDF ou DOCX et retourne son texte brut nettoyé.

    Paramètres
    ----------
    fichier : str — chemin vers le fichier (.pdf ou .docx)

    Retourne
    --------
    str — texte extrait et nettoyé
    """
    if not os.path.isfile(fichier):
        raise FileNotFoundError(f"Fichier introuvable : {fichier}")

    extension = os.path.splitext(fichier)[1].lower()

    if extension == ".pdf":
        texte = _lire_pdf(fichier)
    elif extension == ".docx":
        texte = _lire_docx(fichier)
    else:
        raise ValueError(f"Extension « {extension} » non supportée. Utilisez .pdf ou .docx.")

    return _nettoyer_texte(texte)


print("✅ charger_document() définie.")

✅ charger_document() définie.


### 🧪 Test — `charger_document()`
> Remplace `"mon_fichier.pdf"` par le chemin réel de ton fichier.

In [14]:
CHEMIN_FICHIER = "plant.pdf"   # ← modifie ici

texte_brut = charger_document(CHEMIN_FICHIER)
print(f"📄 Caractères extraits : {len(texte_brut)}")
print("\n--- Aperçu (500 premiers caractères) ---")
print(texte_brut[:500])

📄 Caractères extraits : 44885

--- Aperçu (500 premiers caractères) ---
UNIT 4
The description of structure and variation of living organisms over a
period of time, ended up as two, apparently irreconcilable perspectives
on biology. The two perspectives essentially rested on two levels of
organisation of life forms and phenomena. One described at organismic
and above level of organisation while the second described at cellular
and molecular level of organisation. The first resulted in ecology and
related disciplines. The second resulted in physiology and biochemistr


---
## 2️⃣ Cellule 6 — `decouper_chunks(texte)`
Découpe le texte en morceaux avec chevauchement.

In [15]:
def decouper_chunks(
    texte: str,
    taille_chunk: int = 500,
    chevauchement: int = 100,
) -> List[str]:
    """
    Découpe le texte en morceaux (chunks) avec chevauchement.

    Paramètres
    ----------
    texte         : texte brut issu de charger_document()
    taille_chunk  : nombre de caractères par chunk (défaut 500)
    chevauchement : caractères partagés entre chunks adjacents (défaut 100)

    Retourne
    --------
    List[str] — liste de chunks non vides
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=taille_chunk,
        chunk_overlap=chevauchement,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    chunks = splitter.split_text(texte)
    return [c.strip() for c in chunks if len(c.strip()) > 30]


print("✅ decouper_chunks() définie.")

✅ decouper_chunks() définie.


### 🧪 Test — `decouper_chunks()`

In [16]:
liste_chunks = decouper_chunks(texte_brut)

print(f"✂️  Nombre de chunks : {len(liste_chunks)}")
print(f"📏 Taille moyenne   : {sum(len(c) for c in liste_chunks) // len(liste_chunks)} caractères")
print("\n--- 3 premiers chunks ---")
for i, chunk in enumerate(liste_chunks[:3], 1):
    print(f"\n[Chunk {i}]\n{chunk}")

✂️  Nombre de chunks : 119
📏 Taille moyenne   : 437 caractères

--- 3 premiers chunks ---

[Chunk 1]
UNIT 4
The description of structure and variation of living organisms over a
period of time, ended up as two, apparently irreconcilable perspectives
on biology. The two perspectives essentially rested on two levels of
organisation of life forms and phenomena. One described at organismic
and above level of organisation while the second described at cellular
and molecular level of organisation. The first resulted in ecology and

[Chunk 2]
and molecular level of organisation. The first resulted in ecology and
related disciplines. The second resulted in physiology and biochemistry.
Description of physiological processes, in flowering plants as an
example, is what is given in the chapters in this unit. The processes of
photosynthesis, respiration and ultimately plant growth and
development are described in molecular terms but in the context of
cellular activities and even at organism level. 

---
## 3️⃣ Cellule 8 — `creer_base_vectorielle(chunks)`
Encode les chunks en vecteurs et construit un index FAISS.

In [ ]:
def creer_base_vectorielle(chunks: List[str]) -> dict:
    """
    Encode les chunks en vecteurs et construit un index FAISS.

    Paramètres
    ----------
    chunks : liste de chaînes issue de decouper_chunks()

    Retourne
    --------
    dict :
        - "index"  : faiss.IndexFlatL2
        - "chunks" : List[str]
        - "dim"    : int
    """
    if not chunks:
        raise ValueError("La liste de chunks est vide — aucun document à indexer.")

    model = _get_embed_model()
    print(f"⏳ Encodage de {len(chunks)} chunks…")
    vecteurs = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
    vecteurs = vecteurs.astype(np.float32)

    dim = vecteurs.shape[1]
    index = faiss.IndexFlatL2(dim)# creer faiss
    index.add(vecteurs)

    print(f"✅ Index FAISS créé : {index.ntotal} vecteurs, dimension {dim}")
    return {"index": index, "chunks": chunks, "dim": dim}


print("✅ creer_base_vectorielle() définie.")

✅ creer_base_vectorielle() définie.


### 🧪 Test — `creer_base_vectorielle()`

In [19]:
base_vectorielle = creer_base_vectorielle(liste_chunks)

print(f"\n📊 Résumé de la base vectorielle :")
print(f"   • Vecteurs indexés : {base_vectorielle['index'].ntotal}")
print(f"   • Dimension        : {base_vectorielle['dim']}")
print(f"   • Chunks stockés   : {len(base_vectorielle['chunks'])}")

⏳ Encodage de 119 chunks…


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Index FAISS créé : 119 vecteurs, dimension 384

📊 Résumé de la base vectorielle :
   • Vecteurs indexés : 119
   • Dimension        : 384
   • Chunks stockés   : 119


---
## 4️⃣ Cellule 10 — `rechercher(question, base_vectorielle)`
Trouve les passages les plus pertinents pour une question.

In [20]:
def rechercher(#trouve les passage les plus pertinant pour la question
    question: str,
    base_vectorielle: dict,
    top_k: int = 4,
) -> List[Tuple[str, float]]:
    """
    Trouve les passages les plus pertinents pour une question donnée.

    Paramètres
    ----------
    question         : question posée par l'étudiant (texte libre)
    base_vectorielle : dict retourné par creer_base_vectorielle()
    top_k            : nombre de passages à retourner (défaut 4)

    Retourne
    --------
    List[Tuple[str, float]]
        Liste de (chunk, score_distance) — score faible = très pertinent
    """
    model = _get_embed_model()
    vecteur_q = model.encode([question], convert_to_numpy=True).astype(np.float32)

    index  = base_vectorielle["index"]
    chunks = base_vectorielle["chunks"]

    k_reel = min(top_k, len(chunks))
    distances, indices = index.search(vecteur_q, k_reel)

    resultats = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx != -1:
            resultats.append((chunks[idx], float(dist)))

    return resultats


print("✅ rechercher() définie.")

✅ rechercher() définie.


### 🧪 Test — `rechercher()`
> Modifie la question selon ton document.

In [21]:
MA_QUESTION = "Qu'est-ce que la photosynthèse ?"   # ← modifie ici

resultats = rechercher(MA_QUESTION, base_vectorielle, top_k=3)

print(f"🔍 Question : « {MA_QUESTION} »")
print(f"   {len(resultats)} passages trouvés\n")

for i, (chunk, score) in enumerate(resultats, 1):
    print(f"{'─'*60}")
    print(f"  Résultat #{i}  |  Score L2 = {score:.4f}")
    print(f"{'─'*60}")
    print(chunk[:300] + ("…" if len(chunk) > 300 else ""))
    print()

🔍 Question : « Qu'est-ce que la photosynthèse ? »
   3 passages trouvés

────────────────────────────────────────────────────────────
  Résultat #1  |  Score L2 = 9.4661
────────────────────────────────────────────────────────────
photosynthesis is essentially a light-dependent reaction in which
hydrogen from a suitable oxidisable compound reduces carbon dioxide
to carbohydrates. This can be expressed by:
2
2
2
2
2
2
H A
CO
A
CH O
H O
Light
+

→

+
+
In green plants H2O is the hydrogen donor and is oxidised to O2. Some
o…

────────────────────────────────────────────────────────────
  Résultat #2  |  Score L2 = 10.0037
────────────────────────────────────────────────────────────
PHOTOSYNTHESIS IN HIGHER PLANTS
151
SUMMARY
Green plants make their own food by photosynthesis. During this process carbon
dioxide from the atmosphere is taken in by leaves through stomata and used for
making carbohydrates, principally glucose and starch. Photosynthesis takes place
only in the green…

────

---
## 🔄 Cellule 12 — Pipeline complet (tout en un)
Exécute les 4 étapes d'un coup sur un nouveau fichier.

In [22]:
def pipeline_complet(chemin_fichier: str, question: str, top_k: int = 4):
    """
    Exécute les 4 étapes du pipeline RAG et affiche les résultats.
    Retourne aussi la base vectorielle pour réutilisation.
    """
    print("=" * 60)
    print("ÉTAPE 1 — Chargement du document")
    print("=" * 60)
    texte = charger_document(chemin_fichier)
    print(f"✅ {len(texte)} caractères extraits.\n")

    print("=" * 60)
    print("ÉTAPE 2 — Découpage en chunks")
    print("=" * 60)
    chunks = decouper_chunks(texte)
    print(f"✅ {len(chunks)} chunks créés.\n")

    print("=" * 60)
    print("ÉTAPE 3 — Création de la base vectorielle")
    print("=" * 60)
    base = creer_base_vectorielle(chunks)
    print()

    print("=" * 60)
    print("ÉTAPE 4 — Recherche")
    print("=" * 60)
    resultats = rechercher(question, base, top_k=top_k)
    print(f"🔍 Question : « {question} »\n")
    for i, (chunk, score) in enumerate(resultats, 1):
        print(f"[{i}] Score L2 = {score:.4f}")
        print(chunk[:250] + ("…" if len(chunk) > 250 else ""))
        print()

    return base


# ─── Lance le pipeline ───────────────────────────────────────────────────────
# base = pipeline_complet("mon_fichier.pdf", "Quelle est la définition de X ?", top_k=3)
print("✅ pipeline_complet() définie. Décommente la dernière ligne pour lancer.")

✅ pipeline_complet() définie. Décommente la dernière ligne pour lancer.


---
## 📤 Cellule 13 — Export du module (pour les coéquipiers)
Génère `rag_pipeline.py` directement depuis ce notebook.

In [1]:
import json

notebook_name = "rag_pipeline.ipynb"

with open(notebook_name, "r", encoding="utf-8") as f:
    notebook = json.load(f)

with open("rag_pipeline.py", "w", encoding="utf-8") as f:
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            f.write("".join(cell["source"]))
            f.write("\n\n")

print("✅ rag_pipeline.py créé avec succès.")

✅ rag_pipeline.py créé avec succès.
